# Segmentation of high intensity pixels in TFR of EEG signal

In [86]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram
import pywt
import cv2
from sklearn.decomposition import NMF
from ssqueezepy import ssq_stft, issq_stft
from scipy.linalg import solve

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

## 1. Generate synthetic signal

We generate a synthetic signal. In the EEG we want to mimic the presence of $\delta$ + $\alpha$ bands, as it would be for a gabaergic anesthesia and the addition of a $\beta$ band to mimic the __ketamine signature__.

To generate one band we use a __2D Ornstein-Ulhenbeck__ process that can create spindles like patterns with waning and waxing envelopes containing oscillations.

In [106]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 2]
factor_list = [1, 1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')

axes[1].sharex(axes[0])


plt.show()

Select by hand signal of interest. To be replaced by automatic freqeuncy range segmentation

In [105]:
M = spectro.copy()
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = M[mask_f, :]

fig, axes = plt.subplots(4, sharex = True, constrained_layout = True)
axes[0].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet', label = 'Id')
axes[1].pcolormesh(t_spectro, f_M, M**0.33, shading = 'nearest', cmap = 'jet', label = '**0.33')
axes[2].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet', label = 'log2')
axes[3].pcolormesh(t_spectro, f_M, np.log(M + 1e-11), shading = 'nearest', cmap = 'jet', label = 'log')

plt.show()

# 1. Select high intensity pixels in frequency range of interest

### a. Quantile thresholding

In [82]:
f_int = [25, 35]

mask = M >= np.quantile(M, 0.5) * 3 
freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])
print(np.shape(mask))
mask_f_int = mask * freq_mask[:, None]
print(np.shape(mask_f_int))

fig, axes = plt.subplots(3, 2, sharex = True, constrained_layout = True)
axes[0, 0].set_title('Full mask')
axes[0, 0].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')
axes[1, 0].pcolormesh(t_spectro, f_M, mask, shading = 'nearest', cmap = 'jet')
axes[2, 0].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')
axes[2, 0].contour(t_spectro, f_M, mask, colors = 'white')

axes[0, 1].set_title('Mask cropped to frequency range of interest')
axes[0, 1].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')
axes[1, 1].pcolormesh(t_spectro, f_M, mask_f_int, shading = 'nearest', cmap = 'jet')
axes[2, 1].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')
axes[2, 1].contour(t_spectro, f_M, mask_f_int, colors = 'white')

plt.show()

(50, 201)
(50, 201)


# 2. Extend mask

The above thresholding methods from 1 give a mask that is cropped in f_int. However bins of high intensity pixels that are smoothly connected by smooth gradient of bins intensity to a selected region of the clipped mask should be included, as they form one continuous pattern in time-frequency

### a. Connected components extension using a second less restrictive mask based on bins intensity

The second mask should contain components that are of lower intensity but still high enough not to include background

In [83]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import reconstruction

# --- 1. Your Existing Mask Setup ---
f_int = [25, 35]

mask = M >= np.quantile(M, 0.5) * 3 
freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])
mask_f_int = mask * freq_mask[:, None]

# --- 2. Region Growing / Connected Energy Extension ---
# Seed: Your high-intensity features within f_int
seed_mask = mask_f_int.astype(bool)

# Boundary condition: Allow expansion into any connected bin across the full matrix
# that stays above a noise threshold (e.g., 1.5x median signal, or standard median)
noise_floor = np.quantile(M, 0.5) * 1.5
valid_signal_mask = M >= noise_floor

# Morphological reconstruction (expands seed outward along connected valid_signal_mask)
mask_extended = reconstruction(seed_mask, valid_signal_mask, method='dilation').astype(int)


# --- 3. Plotting & Comparison ---
fig, axes = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(14, 8), constrained_layout=True)

# Column 0: Full initial threshold
axes[0, 0].set_title('Full Spectrogram')
axes[0, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[1, 0].set_title('Full Mask')
axes[1, 0].pcolormesh(t_spectro, f_M, mask, shading='nearest', cmap='jet')
axes[2, 0].set_title('Full Mask Contour')
axes[2, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 0].contour(t_spectro, f_M, mask, colors='white')

# Column 1: Cropped to f_int
axes[0, 1].set_title('Spectrogram + f_int')
axes[0, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 1].axhline(f_int[0], color='white', linestyle='--')
axes[0, 1].axhline(f_int[1], color='white', linestyle='--')
axes[1, 1].set_title('Cropped Mask (mask_f_int)')
axes[1, 1].pcolormesh(t_spectro, f_M, mask_f_int, shading='nearest', cmap='jet')
axes[2, 1].set_title('Cropped Contour')
axes[2, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 1].contour(t_spectro, f_M, mask_f_int, colors='white')

# Column 2: Extended connected mask
axes[0, 2].set_title('Spectrogram + f_int')
axes[0, 2].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 2].axhline(f_int[0], color='white', linestyle='--')
axes[0, 2].axhline(f_int[1], color='white', linestyle='--')
axes[1, 2].set_title('Extended Connected Mask')
axes[1, 2].pcolormesh(t_spectro, f_M, mask_extended, shading='nearest', cmap='jet')
axes[2, 2].set_title('Extended Contour')
axes[2, 2].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 2].contour(t_spectro, f_M, mask_extended, colors='white')

plt.show()

### b. Connected components extension using a second less restrictive mask based on gradient intensity

The second mask should contain components that are of lower intensity but still high enough not to include background

In [84]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import sobel
from skimage.morphology import reconstruction

# --- 1. Your Existing Seed Setup ---
f_int = [25, 35]

mask = M >= np.quantile(M, 0.5) * 3 
freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])
seed_mask = (mask * freq_mask[:, None]).astype(bool)


# --- 2. Gradient-Based Secondary Mask ---
# Compute 2D gradient magnitude using Sobel filters
grad_f = sobel(M, axis=0)  # Gradient along frequencies
grad_t = sobel(M, axis=1)  # Gradient along time
grad_mag = np.hypot(grad_f, grad_t)

# Define gradient and noise thresholds
max_grad_threshold = np.quantile(grad_mag, 0.75) 
smooth_gradient_mask = grad_mag <= max_grad_threshold
noise_floor_mask = M >= np.quantile(M, 0.5)

# Combine criteria
valid_signal_mask = smooth_gradient_mask & noise_floor_mask

# CRITICAL FIX: Ensure seed pixels are ALWAYS present in valid_signal_mask
# (Prevents the ValueError: seed <= mask requirement in skimage)
valid_signal_mask = valid_signal_mask | seed_mask


# --- 3. Morphological Reconstruction ---
mask_grad_extended = reconstruction(seed_mask, valid_signal_mask, method='dilation').astype(int)


# --- 4. Visualizing Gradient & Result ---
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True, constrained_layout=True)

# Top Left: Spectrogram
axes[0, 0].set_title('Spectrogram M')
axes[0, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 0].axhline(f_int[0], color='white', linestyle='--')
axes[0, 0].axhline(f_int[1], color='white', linestyle='--')

# Top Right: Gradient Magnitude
axes[0, 1].set_title('Gradient Magnitude ||∇M||')
im = axes[0, 1].pcolormesh(t_spectro, f_M, grad_mag, shading='nearest', cmap='magma')
fig.colorbar(im, ax=axes[0, 1])

# Bottom Left: Valid Signal Mask
axes[1, 0].set_title('Valid Region Mask (Smooth Gradient)')
axes[1, 0].pcolormesh(t_spectro, f_M, valid_signal_mask, shading='nearest', cmap='binary')

# Bottom Right: Final Extended Mask Contour
axes[1, 1].set_title('Gradient-Extended Contour')
axes[1, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[1, 1].contour(t_spectro, f_M, mask_grad_extended, colors='white')

plt.show()

### c. Extension using smooth gradient direction with Watershed algorithm

This method uses a second mask similar as a. but it expand differently leading to a more restrictive final mask (if second mask is the same).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
from skimage.segmentation import watershed

# --- 1. Your Existing Mask Setup ---
f_int = [25, 35]

mask = M >= np.quantile(M, 0.5) * 3 
freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])
mask_f_int = mask * freq_mask[:, None]

# --- 2. Seeded Watershed Approach ---

# Step A: Label distinct seed regions inside f_int
# Structure (3x3 array of 1s) enables 8-connectivity labeling
structure = np.ones((3, 3), dtype=int)
markers, num_features = label(mask_f_int, structure=structure)

# Step B: Define Topography (Invert M so high intensity peaks = deep valleys)
topography = -M

# Step C: Define a Background Mask / Stop Boundary
# Pixels below this noise floor are treated as "background" (label = 0)
noise_floor = np.quantile(M, 0.5) * 2
background_mask = M < noise_floor

# Copy markers and set the background region to a distinct boundary
watershed_markers = markers.copy()
watershed_markers[background_mask] = -1  # Boundary where flooding stops

# Step D: Run Watershed
# Compactness controls boundary smoothness (0 = standard intensity-driven flooding)
labels = watershed(topography, markers=watershed_markers, mask=~background_mask)

# Convert watershed labels into a binary mask (any pixel assigned a peak label > 0)
mask_watershed = (labels > 0).astype(int)


# --- 3. Plotting Watershed Comparison ---
fig, axes = plt.subplots(3, 2, sharex=True, sharey=True, figsize=(10, 8), constrained_layout=True)

# Left Column: Cropped Mask (Baseline)
axes[0, 0].set_title('Spectrogram + f_int')
axes[0, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 0].axhline(f_int[0], color='white', linestyle='--')
axes[0, 0].axhline(f_int[1], color='white', linestyle='--')

axes[1, 0].set_title('Cropped Mask (mask_f_int)')
axes[1, 0].pcolormesh(t_spectro, f_M, mask_f_int, shading='nearest', cmap='jet')

axes[2, 0].set_title('Cropped Contour')
axes[2, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 0].contour(t_spectro, f_M, mask_f_int, colors='white')

# Right Column: Watershed Extension
axes[0, 1].set_title('Spectrogram + f_int')
axes[0, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 1].axhline(f_int[0], color='white', linestyle='--')
axes[0, 1].axhline(f_int[1], color='white', linestyle='--')

axes[1, 1].set_title('Watershed Extended Mask')
axes[1, 1].pcolormesh(t_spectro, f_M, mask_watershed, shading='nearest', cmap='jet')

axes[2, 1].set_title('Watershed Contour')
axes[2, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 1].contour(t_spectro, f_M, mask_watershed, colors='white')

plt.show()

Up to here we worked on threshold to get threshold for mask of high intensity pixels and threshold separating from background. However we can also get those threhold and use the same methods of threhsolding and expanding with values based on psd baseline.

In [107]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
from skimage.segmentation import watershed

# --- Segmentation using PSD baseline
psd = np.mean(M, axis = 1)
psd_baseline = whittaker_als_baseline(psd, lam=1e4, p=0.01)
factor_high = 3
T_high = factor_high * psd_baseline  # Using your defined factor
mask = M > (T_high[:, np.newaxis]) 

fig, axis = plt.subplots(1)
axis.plot(f_M, psd)
axis.plot(f_M, psd_baseline)

plt.show()

# --- 1. Your Existing Mask Setup ---
f_int = [25, 35]

freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])
mask_f_int = mask * freq_mask[:, None]

# --- 2. Seeded Watershed Approach ---

# Step A: Label distinct seed regions inside f_int
# Structure (3x3 array of 1s) enables 8-connectivity labeling
structure = np.ones((3, 3), dtype=int)
markers, num_features = label(mask_f_int, structure=structure)

# Step B: Define Topography (Invert M so high intensity peaks = deep valleys)
topography = -M

# Step C: Define a Background Mask / Stop Boundary
# Pixels below this noise floor are treated as "background" (label = 0)
factor_low = 2
T_low = factor_low * psd_baseline  # Using your defined factor
background_mask = M < (T_low[:, np.newaxis]) 

# Copy markers and set the background region to a distinct boundary
watershed_markers = markers.copy()
watershed_markers[background_mask] = -1  # Boundary where flooding stops

# Step D: Run Watershed
# Compactness controls boundary smoothness (0 = standard intensity-driven flooding)
labels = watershed(topography, markers=watershed_markers, mask=~background_mask)

# Convert watershed labels into a binary mask (any pixel assigned a peak label > 0)
mask_watershed = (labels > 0).astype(int)

# --- 3. Plotting Watershed Comparison ---
fig, axes = plt.subplots(3, 2, sharex=True, sharey=True, figsize=(10, 8), constrained_layout=True)

# Left Column: Cropped Mask (Baseline)
axes[0, 0].set_title('Spectrogram + f_int')
axes[0, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 0].axhline(f_int[0], color='white', linestyle='--')
axes[0, 0].axhline(f_int[1], color='white', linestyle='--')

axes[1, 0].set_title('Cropped Mask (mask_f_int)')
axes[1, 0].pcolormesh(t_spectro, f_M, mask_f_int, shading='nearest', cmap='jet')

axes[2, 0].set_title('Cropped Contour')
axes[2, 0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 0].contour(t_spectro, f_M, mask_f_int, colors='white')

# Right Column: Watershed Extension
axes[0, 1].set_title('Spectrogram + f_int')
axes[0, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[0, 1].axhline(f_int[0], color='white', linestyle='--')
axes[0, 1].axhline(f_int[1], color='white', linestyle='--')

axes[1, 1].set_title('Watershed Extended Mask')
axes[1, 1].pcolormesh(t_spectro, f_M, mask_watershed, shading='nearest', cmap='jet')

axes[2, 1].set_title('Watershed Contour')
axes[2, 1].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet')
axes[2, 1].contour(t_spectro, f_M, mask_watershed, colors='white')

plt.show()